In [1]:
# =============================================================================
# Cell 1: Setup Environment, Mount Drive, Define Paths
# =============================================================================
import os
import sys
import torch
import gc
import copy
import glob
import random
import json
from collections import defaultdict
import traceback

print("--- Environment Setup ---")

# Set CUDA Launch Blocking (Optional but Recommended for Debugging GPU errors)
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
print("CUDA_LAUNCH_BLOCKING set to 1.")

# Check if in Colab
IN_COLAB = 'google.colab' in sys.modules

# Install necessary libraries
print("Installing required libraries...")
# Using default Colab torch should be fine unless specific version needed
# !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Install other libraries, including torch-pruning
# Updated transformers version might be needed if internal structures change
!pip install -q --upgrade transformers datasets accelerate evaluate timm Pillow safetensors pycocotools thop torch-pruning tqdm
print("Libraries installation attempt finished.")

# Import key libraries (do this after install)
try:
    import torch
    import torch.nn as nn
    import numpy as np
    from transformers import (
        AutoImageProcessor, AutoModelForObjectDetection, AutoConfig, Trainer, TrainingArguments
    )
    # Import MLP class was intended for head recreation fallback (Cell 2), but standard modules are used instead.
    # from transformers.models.deformable_detr.modeling_deformable_detr import DeformableDetrMLP # <<< COMMENTED OUT
    import torchvision
    from tqdm.notebook import tqdm
    from PIL import Image
    from torch.utils.data import Dataset, DataLoader
    import torch_pruning as tp
    from thop import profile
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    print("Core libraries imported successfully.")
except ImportError as e:
    print(f"ERROR: Failed to import libraries: {e}")
    print("Please check the pip install logs above.")
    raise e

# Mount Google Drive if in Colab
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        try:
            drive.mount('/content/drive')
            print("Google Drive mounted.")
        except Exception as e_mount:
            print(f"Error mounting drive: {e_mount}")
            raise e_mount
    else:
        print("Google Drive already mounted.")
    base_drive_path = "/content/drive/MyDrive/"
else:
    base_drive_path = "./" # Adjust if running locally

# --- Configuration ---
print("\n--- Configuration ---")
# !!! Important: Adjust these paths to your actual Drive locations !!!
model_dir = os.path.join(base_drive_path, "deformable-detr-finetuned-kitti") # DIR where fine-tuned model was saved
dataset_base_dir = os.path.join(base_drive_path, "kitti_subset") # Base DIR for KITTI subset
images_dir = os.path.join(dataset_base_dir, "images") # Specific image folder
annotation_dir = os.path.join(dataset_base_dir, "annotations") # Specific annotation folder (for KITTI format)
coco_annotation_file = os.path.join(dataset_base_dir, "annotations.json") # COCO format annotation file (for mAP eval)
output_dir = os.path.join(base_drive_path, "kitti_torch_pruning_output_v1") # Output directory for this run

# Pruning & Fine-tuning Params
GLOBAL_PRUNING_RATIO = 0.1 # Target sparsity for *each* prunable Conv2D layer's channels
DO_FINE_TUNING = True
FINE_TUNE_EPOCHS = 5
FINE_TUNE_LR = 1e-5
FINE_TUNE_BATCH_SIZE = 2 # Keep small for Colab memory
TRAIN_VAL_SPLIT_RATIO = 0.9 # 90% for training, 10% for validation

# Dataset Params (ensure these match your KITTI subset)
NUM_KITTI_CLASSES = 3 # Car, Pedestrian, Cyclist
NUM_OUTPUTS_REQUIRED = NUM_KITTI_CLASSES + 1 # Add 1 for the background/no-object class

# --- End Configuration ---

# Create output directory
os.makedirs(output_dir, exist_ok=True)
print(f"Model directory: {model_dir}")
print(f"Dataset directory: {dataset_base_dir}")
print(f"COCO Annotation file: {coco_annotation_file}")
print(f"Output directory: {output_dir}")

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if not torch.cuda.is_available():
    print("WARNING: CUDA not available, running on CPU. This will be very slow.")

# Helper Functions (if needed by multiple cells, define here)
def get_module_by_name(model: nn.Module, name: str) -> nn.Module:
    """Gets a module from a model using its full name."""
    names = name.split('.')
    obj = model
    for n in names:
        if hasattr(obj, n):
            obj = getattr(obj, n)
        else:
            try: # Handle sequential indexing like layer1.0.conv1
                idx = int(n)
                obj = obj[idx]
            except (ValueError, IndexError, TypeError):
                raise AttributeError(f"Module part '{n}' not found in name '{name}'. Parent type: {type(obj)}")
    return obj

def set_module_by_name(model: nn.Module, name: str, new_module: nn.Module):
    """Sets a module in a model using its full name."""
    names = name.split('.')
    parent_name = '.'.join(names[:-1])
    leaf_name = names[-1]

    try:
        # Use get_submodule which is safer for nested modules
        parent_module = model.get_submodule(parent_name) if parent_name else model
    except AttributeError:
         # Fallback for nested Sequentials/ModuleLists if get_submodule fails
         parent_module = get_module_by_name(model, parent_name)

    if hasattr(parent_module, leaf_name):
        setattr(parent_module, leaf_name, new_module)
    else:
        try: # Handle replacing item in ModuleList/Sequential
            idx = int(leaf_name)
            parent_module[idx] = new_module
        except (ValueError, IndexError, TypeError):
            raise AttributeError(f"Could not set attribute or index '{leaf_name}' in parent module '{parent_name}' of type {type(parent_module)}")

--- Environment Setup ---
CUDA_LAUNCH_BLOCKING set to 1.
Installing required libraries...
Libraries installation attempt finished.
Core libraries imported successfully.
Google Drive already mounted.

--- Configuration ---
Model directory: /content/drive/MyDrive/deformable-detr-finetuned-kitti
Dataset directory: /content/drive/MyDrive/kitti_subset
COCO Annotation file: /content/drive/MyDrive/kitti_subset/annotations.json
Output directory: /content/drive/MyDrive/kitti_torch_pruning_output_v1
Using device: cuda


In [2]:
# =============================================================================
# Cell 2: Load Model, Resize Head, Replace BN (GPU Execution)
# =============================================================================
print("\n--- Loading Model, Resizing Head, Replacing BN ---")

# Ensure variables from Cell 1 are accessible
assert 'model_dir' in locals(), "Cell 1 must be run first to define paths."
assert 'device' in locals(), "Cell 1 must be run first to define device."
assert 'NUM_OUTPUTS_REQUIRED' in locals(), "Cell 1 must define NUM_OUTPUTS_REQUIRED."
assert str(device) == 'cuda', "This cell expects CUDA device for final model placement."

model = None
image_processor = None
config = None
hidden_dim = 256
decoder_layers = 6
num_queries = 300
original_params = -1 # Will be calculated at the end

try:
    # Load processor
    image_processor = AutoImageProcessor.from_pretrained(model_dir)
    print(f"Image processor loaded from {model_dir}")

    # Load config
    try:
        config = AutoConfig.from_pretrained(model_dir)
        num_queries = getattr(config, 'num_queries', 300)
        hidden_dim = getattr(config, 'd_model', 256)
        decoder_layers = getattr(config, 'decoder_layers', 6)
        print(f"Loaded config: num_queries={num_queries}, hidden_dim={hidden_dim}, decoder_layers={decoder_layers}")
        id2label = {i: f"LABEL_{i}" for i in range(NUM_OUTPUTS_REQUIRED)}
        label2id = {v: k for k, v in id2label.items()}
        config.id2label = id2label; config.label2id = label2id; config.num_labels = NUM_OUTPUTS_REQUIRED
        print(f"Updated config for {NUM_OUTPUTS_REQUIRED} outputs (incl. background).")
    except Exception as e_conf:
        print(f"WARNING: Could not load/parse config: {e_conf}. Using defaults.")
        id2label = {i: f"LABEL_{i}" for i in range(NUM_OUTPUTS_REQUIRED)}
        label2id = {v: k for k, v in id2label.items()}
        config = AutoConfig.from_pretrained("SenseTime/deformable-detr", num_labels=NUM_OUTPUTS_REQUIRED, id2label=id2label, label2id=label2id)
        num_queries = getattr(config, 'num_queries', num_queries); hidden_dim = getattr(config, 'd_model', hidden_dim); decoder_layers = getattr(config, 'decoder_layers', decoder_layers)
        print(f"Using potentially default config values: num_queries={num_queries}, hidden_dim={hidden_dim}, decoder_layers={decoder_layers}")

    # Load model structure (on CPU initially)
    print("Loading model structure (ignore mismatched sizes)...")
    model = AutoModelForObjectDetection.from_pretrained(
        model_dir,
        config=config,
        ignore_mismatched_sizes=True
    )
    print("Model structure loaded (on CPU initially).")

    # --- Resize Head AFTER Loading (on CPU) ---
    print("Checking and potentially resizing model heads (on CPU)...")
    try:
        current_cls_outputs = -1; final_class_layer = None
        if hasattr(model, 'class_embed') and isinstance(model.class_embed, nn.ModuleList) and len(model.class_embed) > 0:
             last_mod_in_list = model.class_embed[-1]
             if isinstance(last_mod_in_list, nn.Linear): final_class_layer = last_mod_in_list
             elif hasattr(last_mod_in_list, 'layers') and isinstance(last_mod_in_list.layers, nn.Sequential):
                 if len(last_mod_in_list.layers) > 0 and isinstance(last_mod_in_list.layers[-1], nn.Linear): final_class_layer = last_mod_in_list.layers[-1]
        elif hasattr(model, 'class_embed') and isinstance(model.class_embed, nn.Sequential): # ... (other checks) ...
            if len(model.class_embed) > 0 and isinstance(model.class_embed[-1], nn.Linear): final_class_layer = model.class_embed[-1]
        elif hasattr(model, 'class_embed') and isinstance(model.class_embed, nn.Linear): final_class_layer = model.class_embed

        if isinstance(final_class_layer, nn.Linear): current_cls_outputs = final_class_layer.out_features; print(f"  Detected {current_cls_outputs} outputs in loaded classification head.")
        else: print(f"  Could not reliably detect output features of class_embed (Type: {type(model.class_embed)}). Assuming resize needed."); current_cls_outputs = -1

        if current_cls_outputs != NUM_OUTPUTS_REQUIRED:
            print(f"  Head mismatch/uncertainty: Recreating heads for {decoder_layers} layers with {NUM_OUTPUTS_REQUIRED} outputs (CPU).")
            if hasattr(config, 'decoder_layers'): # ... (Recreate heads based on config) ...
                model.class_embed = nn.ModuleList([nn.Linear(hidden_dim, NUM_OUTPUTS_REQUIRED) for _ in range(config.decoder_layers)])
                bbox_head_mlp = lambda: nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 4))
                model.bbox_embed = nn.ModuleList([bbox_head_mlp() for _ in range(config.decoder_layers)])
            else: # ... (Fallback) ...
                 model.class_embed = nn.Linear(hidden_dim, NUM_OUTPUTS_REQUIRED); model.bbox_embed = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 4))
            # ... (Query embedding check/resize) ...
            if hasattr(model, 'query_position_embeddings') and isinstance(model.query_position_embeddings, nn.Embedding):
                 if model.query_position_embeddings.num_embeddings != num_queries: print(f"  Resizing query embeds."); model.query_position_embeddings = nn.Embedding(num_queries, hidden_dim)
                 else: print("  Query embeds size OK.")
            else: print("  Creating query embeds."); model.query_position_embeddings = nn.Embedding(num_queries, hidden_dim)
            print("  Heads recreated/resized on CPU.")
        else: print("  Loaded model heads appear to have the correct number of outputs.")
    except Exception as e_head: print(f"  Error during head check/resize: {e_head}"); traceback.print_exc(); raise RuntimeError("Failed head check/resize.") from e_head

    # --- Replace FrozenBatchNorm2d (on CPU) ---
    print("\nReplacing FrozenBatchNorm2d layers (on CPU)...")
    FROZEN_BN_TYPE = None
    try: # Find the FrozenBN class
        from transformers.models.deformable_detr.modeling_deformable_detr import DeformableDetrFrozenBatchNorm2d
        FROZEN_BN_TYPE = DeformableDetrFrozenBatchNorm2d
        print("  Found DeformableDetrFrozenBatchNorm2d class.")
    except ImportError: print("  DeformableDetrFrozenBatchNorm2d not found.") # Add fallback if needed

    if FROZEN_BN_TYPE:
        replacement_count = 0; error_count = 0
        module_list = list(model.named_modules())
        print(f"  Iterating through {len(module_list)} modules...")
        from tqdm import tqdm as regular_tqdm # Use standard tqdm
        for name, module in regular_tqdm(module_list, desc="Replacing FrozenBN (CPU)", leave=False):
            if isinstance(module, FROZEN_BN_TYPE):
                try: # Replace with standard BN
                    if hasattr(module, 'weight') and module.weight is not None: num_features = module.weight.shape[0]
                    else: print(f"    WARNING: Skipping {name} - no weight attr."); error_count += 1; continue
                    # --- Use eps=1e-5 fix ---
                    new_bn = nn.BatchNorm2d(num_features, eps=1e-5, affine=True, track_running_stats=True)
                    # --- Copy parameters ---
                    if hasattr(module, 'weight') and module.weight is not None and hasattr(new_bn,'weight') and new_bn.weight.shape == module.weight.shape: new_bn.weight.data.copy_(module.weight.data)
                    if hasattr(module, 'bias') and module.bias is not None and hasattr(new_bn,'bias') and new_bn.bias.shape == module.bias.shape: new_bn.bias.data.copy_(module.bias.data)
                    if hasattr(module, 'running_mean') and module.running_mean is not None and hasattr(new_bn,'running_mean') and new_bn.running_mean.shape == module.running_mean.shape: new_bn.running_mean.data.copy_(module.running_mean.data)
                    if hasattr(module, 'running_var') and module.running_var is not None and hasattr(new_bn,'running_var') and new_bn.running_var.shape == module.running_var.shape: new_bn.running_var.data.copy_(module.running_var.data)
                    if hasattr(module, 'num_batches_tracked') and module.num_batches_tracked is not None and hasattr(new_bn, 'num_batches_tracked'): new_bn.num_batches_tracked.data.copy_(module.num_batches_tracked.data)
                    set_module_by_name(model, name, new_bn)
                    replacement_count += 1
                except Exception as e_replace: print(f"    ERROR replacing {name}: {e_replace}"); traceback.print_exc(); error_count += 1
        print(f"  Finished FrozenBN Replacement. Replaced: {replacement_count}, Errors: {error_count}")
        if error_count > 0: raise RuntimeError("Errors occurred during BatchNorm replacement.")
    else: print("  No FrozenBatchNorm class identified for replacement.")

    # --- Move final model (with std BN) to target device (GPU) ---
    print(f"\nMoving final prepared model to: {device}")
    model.to(device)
    model.eval()
    print("Model is ready on device.")

    # --- Calculate original params *after* prep and move to GPU ---
    original_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model parameters after prep (trainable): {original_params:,}")
    # Make variable available globally for Cell 4
    # (Alternatively, Cell 4 could access model.parameters() directly)
    globals()['original_params'] = original_params

except Exception as e_load_prep:
    print(f"ERROR during model loading/preparation: {e_load_prep}")
    traceback.print_exc(); model = None; raise e_load_prep

# Clean up memory
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.



--- Loading Model, Resizing Head, Replacing BN ---
Image processor loaded from /content/drive/MyDrive/deformable-detr-finetuned-kitti
Loaded config: num_queries=300, hidden_dim=256, decoder_layers=6
Updated config for 4 outputs (incl. background).
Loading model structure (ignore mismatched sizes)...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:2397: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:2397: UserWarning: for bn1.wei

Model structure loaded (on CPU initially).
Checking and potentially resizing model heads (on CPU)...
  Detected 4 outputs in loaded classification head.
  Loaded model heads appear to have the correct number of outputs.

Replacing FrozenBatchNorm2d layers (on CPU)...
  Found DeformableDetrFrozenBatchNorm2d class.
  Iterating through 425 modules...


  Finished FrozenBN Replacement. Replaced: 53, Errors: 0

Moving final prepared model to: cuda
Model is ready on device.
Model parameters after prep (trainable): 39,878,026


In [3]:
# =============================================================================
# Cell 3: Calculate Original Metrics (GPU Execution)
# =============================================================================
print("\n--- Calculating Original Model Metrics ---\n")

# Ensure model and param count from Cell 2 are available
assert 'model' in locals() and model is not None, "Cell 2 must be run first."
assert 'original_params' in locals() and original_params > 0, "Cell 2 must set original_params > 0."
assert str(next(model.parameters()).device) == 'cuda:0', "Model expected on CUDA device." # Check device

original_gflops = 0.0
dummy_input = None

if model is not None:
    print(f"Using Original Trainable Parameters (Post-BN Replace): {original_params:,}")

    # Calculate GFLOPs using thop
    bs = 1
    try:
         # Determine input size
         img_h, img_w = 800, 1333
         if hasattr(image_processor, 'size') and isinstance(image_processor.size, dict):
            # ... (Robust size calculation logic) ...
            size_dict = image_processor.size
            if 'shortest_edge' in size_dict:
                shortest = size_dict['shortest_edge']; max_size = getattr(image_processor, 'max_size', 1333); aspect_ratio = 1333 / 800
                if shortest == 800 and max_size == 1333: img_h, img_w = 800, 1333
                else: img_h = shortest; img_w = int(shortest * aspect_ratio);
                if img_w > max_size: img_w = max_size; img_h = int(max_size / aspect_ratio)
            elif 'height' in size_dict and 'width' in size_dict: img_h = size_dict['height']; img_w = size_dict['width']
            img_h = max(img_h, 32); img_w = max(img_w, 32)
         print(f"Using dummy input size H={img_h}, W={img_w}")

         # --- Create dummy input on GPU ---
         dummy_input = torch.randn(bs, 3, img_h, img_w, device=device)
         print(f"Using dummy input shape for GFLOPs: {dummy_input.shape} on {device}")

         # Profile on GPU
         print(f"Calculating GFLOPs on {device} (thop)...")
         try:
              # Ensure model is on GPU (should be already)
              model.to(device)
              flops, params_thop = profile(model, inputs=(dummy_input,), verbose=False)
              original_gflops = flops / 1e9
              print(f"GFLOPs calculated on {device}: {original_gflops:.2f} GFLOPs")
         except Exception as e_prof_gpu:
              print(f"  Thop profile on GPU failed ({e_prof_gpu}), trying on CPU...")
              # --- Fallback to CPU profiling ---
              try:
                   model_cpu_copy = copy.deepcopy(model).cpu()
                   dummy_input_cpu = dummy_input.cpu()
                   flops, _ = profile(model_cpu_copy, inputs=(dummy_input_cpu,), verbose=False)
                   original_gflops = flops / 1e9
                   print(f"GFLOPs calculated on CPU: {original_gflops:.2f} GFLOPs")
                   del model_cpu_copy, dummy_input_cpu; gc.collect()
              except Exception as e_prof_cpu:
                   print(f"  Thop profile on CPU also failed: {e_prof_cpu}")
                   original_gflops = -1.0 # Indicate failure
               # --- End Fallback ---

    except Exception as e_metrics:
        print(f"Error during original metrics calculation: {e_metrics}")
        traceback.print_exc(); original_gflops = -1.0
else:
    print("Model not loaded, cannot calculate original metrics.")
    original_gflops = -1.0

# Store dummy_input globally if needed by Cell 4, otherwise delete
# del dummy_input # Keep dummy_input for Cell 4
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


--- Calculating Original Model Metrics ---

Using Original Trainable Parameters (Post-BN Replace): 39,878,026
Using dummy input size H=800, W=1333
Using dummy input shape for GFLOPs: torch.Size([1, 3, 800, 1333]) on cuda
Calculating GFLOPs on cuda (thop)...
GFLOPs calculated on cuda: 204.87 GFLOPs


In [4]:
# =============================================================================
# Cell 4: Pruning with torch-pruning (GPU Execution)
# =============================================================================
print("\n--- Pruning Model with torch-pruning (GPU Execution) ---\n")

model_pruned = None # Initialize
original_params = -1 # Initialize, will be calculated below

# --- Prerequisite Checks ---
if 'model' not in locals() or model is None:
    print("Original model ('model') not available. Skipping pruning.")
elif not torch.cuda.is_available():
     # This shouldn't happen now, but good practice to keep check
    print("CUDA not available. Skipping GPU pruning.")
else:
    try:
        # --- Calculate original params directly from model object ---
        print("Calculating original params from model object...")
        assert 'model' in locals() and model is not None, "Model not loaded from Cell 2"
        # Ensure model is on the correct device before counting
        device = next(model.parameters()).device
        print(f"  Model is on device: {device}")
        original_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        assert original_params > 0, "Could not calculate positive original_params from model"
        print(f"Using original trainable params (Post-BN Replace): {original_params:,}")
        # --- End Param Calculation ---

        # 1. Create a deep copy directly on GPU
        print(f"\nCreating deepcopy on device ({device})...")
        # Ensure original model is on the correct device (redundant check, but safe)
        model.to(device)
        model_pruned = copy.deepcopy(model)
        model_pruned.eval()
        print("Deepcopy created on target device.")

        # --- BN Replacement is done in Cell 2 now ---
        print("\nSkipping BN Replacement (should be done in Cell 2).")

        # --- Model Structure Debugging (Optional but can keep) ---
        print("\n--- Debugging Model Structure ---")
        print(f"Model pruned is on device: {next(model_pruned.parameters()).device}")
        print("Top-level modules in model_pruned:")
        for name, module in model_pruned.named_children(): print(f"  - {name} ({type(module).__name__})")
        if hasattr(model_pruned, 'model'):
             nested_model_debug = model_pruned.model # Define for clarity
             print("\nModules in model_pruned.model (nested):")
             for name, module in nested_model_debug.named_children(): print(f"  - model.{name} ({type(module).__name__})")
             if hasattr(nested_model_debug, 'backbone'):
                  print("\nModules in model_pruned.model.backbone:")
                  for name, module in nested_model_debug.backbone.named_children(): print(f"  - model.backbone.{name} ({type(module).__name__})")

        # ... Add more detailed backbone checks if needed ...
        print("--- End Debugging ---")

        # 2. Define Dummy Input on GPU
        # Reuse from Cell 3 if possible, otherwise recreate
        if 'dummy_input' not in locals() or dummy_input is None:
             print("\nDummy input not found, recreating on GPU...")
             bs = 1; img_h, img_w = 800, 1333
             try: # Simplified size logic
                 if hasattr(image_processor, 'size') and isinstance(image_processor.size, dict):
                    size_dict = image_processor.size
                    if 'shortest_edge' in size_dict: shortest = size_dict['shortest_edge']; max_size = getattr(image_processor, 'max_size', 1333); img_h = shortest; img_w = int(shortest * (1333/800)); img_w = min(img_w, max_size)
                    elif 'height' in size_dict and 'width' in size_dict: img_h = size_dict['height']; img_w = size_dict['width']
                 img_h = max(img_h, 32); img_w = max(img_w, 32)
                 print(f"Determined dummy input size: H={img_h}, W={img_w}")
             except Exception as e_size: print(f"Warning: Size error: {e_size}. Using default."); img_h, img_w = 800, 1333
             dummy_input = torch.randn(bs, 3, img_h, img_w, device=device) # Create on GPU device
             print(f"Recreated GPU dummy input: {dummy_input.shape} on {dummy_input.device}")
        elif dummy_input.device != device:
             print(f"\nMoving existing dummy input to {device}"); dummy_input = dummy_input.to(device)
        else:
             print(f"\nUsing existing dummy input: {dummy_input.shape} on {dummy_input.device}")


        # 3. Define Ignored Layers (Using Corrected Nested Access)
        ignored_layers_modules = []
        print("\nIdentifying layers to ignore...")
        if not hasattr(model_pruned, 'model') or not isinstance(model_pruned.model, nn.Module):
             raise AttributeError("Nested 'model' module not found in model_pruned")
        else:
             nested_model = model_pruned.model # Use the nested model for backbone checks
             print(f"  Accessing nested model of type: {type(nested_model).__name__}")
             def add_module_and_submodules(module_instance, name_prefix=""):
                  if module_instance is None or not isinstance(module_instance, nn.Module): return
                  if module_instance not in ignored_layers_modules: ignored_layers_modules.append(module_instance)
                  for submodule in module_instance.modules():
                      if submodule not in ignored_layers_modules and submodule is not module_instance: ignored_layers_modules.append(submodule)

             # A. Ignore backbone's initial conv and bn
             if hasattr(nested_model, 'backbone') and hasattr(nested_model.backbone, 'conv_encoder') and hasattr(nested_model.backbone.conv_encoder, 'model'):
                 timm_model = nested_model.backbone.conv_encoder.model; print("  Checking backbone conv1/bn1...")
                 if hasattr(timm_model, 'conv1') and isinstance(timm_model.conv1, nn.Conv2d): ignored_layers_modules.append(timm_model.conv1); print(f"    Added model.backbone.conv1")
                 # Use standard BN type here as it should have been replaced in Cell 2
                 if hasattr(timm_model, 'bn1') and isinstance(timm_model.bn1, (nn.BatchNorm2d, nn.SyncBatchNorm)): ignored_layers_modules.append(timm_model.bn1); print(f"    Added model.backbone.bn1")

                 # B. Ignore backbone's downsample layers
                 print("  Checking backbone downsample blocks...")
                 for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
                      if hasattr(timm_model, layer_name):
                          res_layer = getattr(timm_model, layer_name)
                          if hasattr(res_layer, '__iter__'):
                              for block_idx, block in enumerate(res_layer):
                                   if hasattr(block, 'downsample') and block.downsample is not None: print(f"    Adding downsample block: model.backbone.{layer_name}.{block_idx}.downsample"); add_module_and_submodules(block.downsample, f"ds_{layer_name}_{block_idx}")

             # C. Ignore heads (parallel to nested_model)
             print("  Checking heads, level_embed, input_proj...")
             if hasattr(model_pruned, 'class_embed'): print("    Adding class_embed and submodules"); add_module_and_submodules(model_pruned.class_embed, "class_embed")
             if hasattr(model_pruned, 'bbox_embed'): print("    Adding bbox_embed and submodules"); add_module_and_submodules(model_pruned.bbox_embed, "bbox_embed")

             # D. Ignore level_embed (inside nested_model)
             if hasattr(nested_model, 'level_embed') and isinstance(nested_model.level_embed, nn.Embedding): ignored_layers_modules.append(nested_model.level_embed); print("    Added model.level_embed")

             # E. Ignore input projection (inside nested_model)
             if hasattr(nested_model, 'input_proj'): print("    Adding model.input_proj and submodules"); add_module_and_submodules(nested_model.input_proj, "input_proj")

             # F. No need to check for FrozenBN here
             print("  Skipping check for FrozenBN (should be replaced in Cell 2).")

        ignored_layers = list(set(m for m in ignored_layers_modules if isinstance(m, nn.Module) and m is not model_pruned and (not 'nested_model' in locals() or m is not nested_model)))
        print(f"\nIdentified {len(ignored_layers)} unique nn.Module instances to ignore.")
        print("--- Ignored Modules ---")
        from collections import Counter
        type_counts = Counter(type(m).__name__ for m in ignored_layers)
        print(f"Ignored layer type counts: {type_counts}")
        print("-----------------------")


        # 4. Define Importance and Pruner (on GPU)
        print(f"\nSetting up Pruner on GPU with layer channel sparsity target: {GLOBAL_PRUNING_RATIO}")
        importance = tp.importance.MagnitudeImportance(p=1)

        pruner = tp.pruner.MagnitudePruner(
            model_pruned,               # On GPU, has standard BN
            example_inputs=dummy_input, # On GPU
            importance=importance,
            pruning_ratio=GLOBAL_PRUNING_RATIO, # Use ratio from Cell 1
            ignored_layers=ignored_layers,
            root_module_types=[nn.Conv2d],
            round_to=8,
        )

        # 5. Apply Pruning (on GPU)
        print("Applying pruner.step() on GPU...")
        pruner.step() # Execute on GPU
        print("torch-pruning step finished on GPU.")
        model_pruned.eval()

        print(f"Pruned model ready on {next(model_pruned.parameters()).device}.")

    except Exception as e_prune:
        print(f"ERROR during torch-pruning: {e_prune}")
        traceback.print_exc()
        model_pruned = None

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


--- Pruning Model with torch-pruning (GPU Execution) ---

Calculating original params from model object...
  Model is on device: cuda:0
Using original trainable params (Post-BN Replace): 39,878,026

Creating deepcopy on device (cuda:0)...
Deepcopy created on target device.

Skipping BN Replacement (should be done in Cell 2).

--- Debugging Model Structure ---
Model pruned is on device: cuda:0
Top-level modules in model_pruned:
  - model (DeformableDetrModel)
  - class_embed (ModuleList)
  - bbox_embed (ModuleList)

Modules in model_pruned.model (nested):
  - model.backbone (DeformableDetrConvModel)
  - model.input_proj (ModuleList)
  - model.query_position_embeddings (Embedding)
  - model.encoder (DeformableDetrEncoder)
  - model.decoder (DeformableDetrDecoder)
  - model.reference_points (Linear)

Modules in model_pruned.model.backbone:
  - model.backbone.conv_encoder (DeformableDetrConvEncoder)
  - model.backbone.position_embedding (DeformableDetrSinePositionEmbedding)
--- End Debugg

/usr/local/lib/python3.11/dist-packages/torch_pruning/dependency.py:699: UserWarning: Unwrapped parameters detected: ['model.level_embed'].
 Torch-Pruning will prune the last non-singleton dimension of these parameters. If you wish to change this behavior, please provide an unwrapped_parameters argument.
  warnings.warn(warning_str)


Applying pruner.step() on GPU...
torch-pruning step finished on GPU.
Pruned model ready on cuda:0.


In [5]:
# =============================================================================
# Cell 5: Calculate Pruned Model Metrics
# =============================================================================
print("\n--- Calculating Pruned Model Metrics ---")

pruned_params = -1 # Initialize
pruned_gflops = -1.0 # Initialize

if 'model_pruned' in locals() and model_pruned is not None:
    try:
        # Ensure model is on the correct device for calculations
        model_pruned.to(device)
        model_pruned.eval()

        pruned_params = sum(p.numel() for p in model_pruned.parameters() if p.requires_grad)
        print(f"Pruned Trainable Parameters: {pruned_params:,}")
        if 'original_params' in locals() and original_params > 0:
            reduction = (original_params - pruned_params) / original_params * 100
            print(f"Parameter Reduction: {reduction:.2f}%")
        else:
            print("Cannot calculate reduction (original_params unavailable).")

        # Calculate GFLOPs using thop
        if 'dummy_input' not in locals() or dummy_input is None:
             print("Dummy input not found from Cell 3, recreating...")
             bs = 1; height, width = 800, 1333
             dummy_input = torch.randn(bs, 3, height, width, device=device)
             print(f"Recreated dummy input shape: {dummy_input.shape}")
        elif dummy_input.device != device: # Ensure dummy input is on the correct device
             dummy_input = dummy_input.to(device)

        print("Calculating pruned GFLOPs...")
        try:
            flops, params_thop = profile(model_pruned, inputs=(dummy_input,), verbose=False)
            pruned_gflops = flops / 1e9
            print(f"Pruned GFLOPs calculated on {device}: {pruned_gflops:.2f} GFLOPs")
        except Exception as e_prof_gpu:
            print(f"  Thop profile on GPU failed ({e_prof_gpu}), trying on CPU...")
            try:
                model_pruned_cpu = copy.deepcopy(model_pruned).cpu()
                dummy_input_cpu = dummy_input.cpu()
                flops, _ = profile(model_pruned_cpu, inputs=(dummy_input_cpu,), verbose=False)
                pruned_gflops = flops / 1e9
                print(f"Pruned GFLOPs calculated on CPU: {pruned_gflops:.2f} GFLOPs")
                del model_pruned_cpu, dummy_input_cpu
                gc.collect()
            except Exception as e_prof_cpu:
                print(f"  Thop profile on CPU also failed: {e_prof_cpu}")
                pruned_gflops = -1.0

        if 'original_gflops' in locals() and original_gflops > 0 and pruned_gflops >= 0:
             gflops_reduction = (original_gflops - pruned_gflops) / original_gflops * 100
             print(f"GFLOPs Reduction: {gflops_reduction:.2f}%")
        elif 'original_gflops' in locals() and original_gflops <= 0:
             print("Cannot calculate GFLOPs reduction (original_gflops unavailable).")

    except Exception as e_metrics:
        print(f"Error calculating pruned metrics: {e_metrics}")
        traceback.print_exc()
else:
    print("Pruned model not available, cannot calculate metrics.")


--- Calculating Pruned Model Metrics ---
Pruned Trainable Parameters: 36,535,818
Parameter Reduction: 8.38%
Calculating pruned GFLOPs...
Pruned GFLOPs calculated on cuda:0: 191.19 GFLOPs
GFLOPs Reduction: 6.68%


In [6]:
# =============================================================================
# Cell 6: Save Pruned Model Structure
# =============================================================================
print("\n--- Saving Pruned Model Structure ---")

pruned_model_saved_path = None
if 'model_pruned' in locals() and model_pruned is not None:
    try:
        save_filename = f"ddetr_torchpruned_ratio{GLOBAL_PRUNING_RATIO:.1f}_structure.safetensors"
        pruned_model_saved_path = os.path.join(output_dir, save_filename)
        print(f"Attempting to save pruned structure to: {pruned_model_saved_path}")

        # Use Hugging Face save_pretrained for better compatibility if structure changed significantly
        model_pruned.save_pretrained(output_dir) # Saves config, weights etc. in the output dir
        print(f"Pruned model saved using save_pretrained to: {output_dir}")
        # We will use this directory for loading later if needed

        # Optional: Also save just the state dict if preferred
        # from safetensors.torch import save_file
        # model_pruned.cpu()
        # save_file(model_pruned.state_dict(), pruned_model_saved_path)
        # model_pruned.to(device)
        # print("Pruned model state_dict saved successfully (.safetensors).")

    except Exception as e_save:
        print(f"Error saving pruned model: {e_save}")
        traceback.print_exc()
else:
    print("Pruned model not available, skipping save.")


--- Saving Pruned Model Structure ---
Attempting to save pruned structure to: /content/drive/MyDrive/kitti_torch_pruning_output_v1/ddetr_torchpruned_ratio0.1_structure.safetensors
Pruned model saved using save_pretrained to: /content/drive/MyDrive/kitti_torch_pruning_output_v1


In [17]:
# =============================================================================
# Cell 7: Prepare KITTI Dataset for Fine-tuning / Evaluation
# =============================================================================
print("\n--- Preparing KITTI Dataset ---")

# Keep DefaultDataCollator import just in case, but we likely won't use it
from transformers import DefaultDataCollator
import traceback # Ensure traceback is imported

train_dataloader = None
val_dataloader = None
kitti_dataset_full_coco_fmt = None
coco_gt = None

# --- File existence checks ---
if not os.path.isdir(images_dir):
    print(f"ERROR: Image directory not found: {images_dir}")
    raise FileNotFoundError(f"Image directory not found: {images_dir}")
if not os.path.exists(coco_annotation_file):
     print(f"WARNING: COCO annotation file not found: {coco_annotation_file}. mAP evaluation will not work.")
if not os.path.isdir(annotation_dir):
     print(f"WARNING: KITTI annotation directory not found: {annotation_dir}.")
     if DO_FINE_TUNING: # Only disable if planning to fine-tune
         print("Disabling fine-tuning because KITTI annotations are missing.")
         DO_FINE_TUNING = False
# --- End checks ---


# --- Define Dataset Class (KittiObjectDetectionDataset) ---
class KittiObjectDetectionDataset(Dataset):
    def __init__(self, image_paths, annotation_dir, image_processor, label2id):
        self.image_paths = [p for p in image_paths if os.path.exists(p)]
        self.annotation_dir = annotation_dir
        self.image_processor = image_processor
        self.label2id = {k.lower(): v for k, v in label2id.items() if isinstance(k, str)}
        self.id2label = {v: k for k, v in self.label2id.items()}
        print(f"Dataset Initialized. Found {len(self.image_paths)} images. Category map: {self.label2id}")
        if not self.label2id: print("WARNING: label2id map seems incorrect or empty.")
        if not os.path.isdir(self.annotation_dir): print(f"WARNING: Annotation directory missing: {self.annotation_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            image = Image.open(img_path).convert("RGB")
            img_w, img_h = image.size
            ann_file_name = os.path.splitext(os.path.basename(img_path))[0] + ".txt"
            ann_file_path = os.path.join(self.annotation_dir, ann_file_name)
            annotations_coco_fmt = {"image_id": idx, "annotations": []}
            # --- Parse KITTI annotations ---
            if os.path.exists(ann_file_path):
                with open(ann_file_path, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if not parts: continue
                        cat_name = parts[0].lower()
                        if cat_name in self.label2id:
                            try:
                                bbox_kitti = [float(x) for x in parts[4:8]]
                                xmin, ymin, xmax, ymax = bbox_kitti
                                xmin_c, ymin_c = max(0., xmin), max(0., ymin)
                                xmax_c, ymax_c = min(float(img_w - 1), xmax), min(float(img_h - 1), ymax)
                                if xmin_c < xmax_c and ymin_c < ymax_c:
                                    box_w, box_h = xmax_c - xmin_c, ymax_c - ymin_c
                                    annotations_coco_fmt["annotations"].append({
                                        "image_id": idx, "category_id": self.label2id[cat_name],
                                        "bbox": [xmin_c, ymin_c, box_w, box_h], "area": box_w * box_h, "iscrowd": 0,
                                    })
                            except (ValueError, IndexError) as e_parse: continue
            # --- End Parse ---
            # --- Preprocess using image_processor ---
            try: encoding = self.image_processor(images=image, annotations=annotations_coco_fmt, return_tensors="pt")
            except Exception as e_proc:
                 print(f"\nERROR during image_processor call for {img_path}: {e_proc}")
                 try: # Try image only
                      print("  Trying image processing without annotations...")
                      encoding = self.image_processor(images=image, return_tensors="pt")
                      encoding['labels'] = [{'boxes': torch.empty((0, 4)), 'class_labels': torch.empty((0,), dtype=torch.long)}]
                      print("  Image processed, using dummy labels.")
                 except Exception as e_img_proc: print(f"  ERROR processing even just the image: {e_img_proc}"); return None
            pixel_values = encoding["pixel_values"].squeeze(0)
            labels = encoding["labels"][0] if isinstance(encoding.get("labels"), list) and len(encoding["labels"]) > 0 else {'boxes': torch.empty((0, 4)), 'class_labels': torch.empty((0,), dtype=torch.long)}
            # --- Return dictionary with tensors ---
            return {"pixel_values": pixel_values, "labels": labels}
        except Exception as e: print(f"\nERROR processing sample index {idx} ({img_path}): {e}"); traceback.print_exc(); return None
# --- End Dataset Class ---


# --- Create Datasets and Dataloaders for Training/Validation ---
if DO_FINE_TUNING:
    print("\nCreating training and validation datasets...")
    all_image_files = sorted(glob.glob(os.path.join(images_dir,"**","*.[pP][nN][gG]"), recursive=True) + glob.glob(os.path.join(images_dir,"**","*.[jJ][pP][gG]"), recursive=True) + glob.glob(os.path.join(images_dir,"**","*.[jJ][pP][eE][gG]"), recursive=True))
    if not all_image_files: print(f"ERROR: No images found in {images_dir}. Cannot fine-tune."); DO_FINE_TUNING = False
    else:
        print(f"Found {len(all_image_files)} total images.")
        random.seed(42); random.shuffle(all_image_files)
        split_idx = int(len(all_image_files) * TRAIN_VAL_SPLIT_RATIO); train_image_files = all_image_files[:split_idx]; val_image_files = all_image_files[split_idx:]
        if not val_image_files and train_image_files: print("Warning: No validation files after split, moving one from train."); val_image_files.append(train_image_files.pop())
        if not train_image_files or not val_image_files: print("Error: Could not create non-empty train/val splits. Disabling fine-tuning."); DO_FINE_TUNING = False
        else:
             print(f"Using {len(train_image_files)} train and {len(val_image_files)} validation images.")
             try:
                  # --- label2id setup ---
                  kitti_label2id_from_config = getattr(config, 'label2id', None)
                  if isinstance(kitti_label2id_from_config, dict) and len(kitti_label2id_from_config) == NUM_KITTI_CLASSES: kitti_label2id = kitti_label2id_from_config; print("Using label2id map from loaded config.")
                  else:
                       print(f"Warning: Config label2id map missing/invalid/wrong size. Creating default map.")
                       kitti_cats = ['Car', 'Pedestrian', 'Cyclist']; assert NUM_KITTI_CLASSES == len(kitti_cats), "Class num mismatch"
                       kitti_label2id = {name: i for i, name in enumerate(kitti_cats)}
                  print(f"Using label2id map for dataset: {kitti_label2id}")
                  assert 'image_processor' in locals() and image_processor is not None, "Image processor not loaded."
                  # --- End label2id setup ---

                  train_dataset = KittiObjectDetectionDataset(train_image_files, annotation_dir, image_processor, kitti_label2id)
                  val_dataset = KittiObjectDetectionDataset(val_image_files, annotation_dir, image_processor, kitti_label2id)

                  # --- Define the Custom Collator (Revised) ---
                  def custom_object_detection_collator(batch):
                      batch = [item for item in batch if item is not None]
                      if not batch: return None
                      pixel_values = [item["pixel_values"] for item in batch]
                      labels = [item["labels"] for item in batch]
                      try:
                          # --- Pass list directly to pad ---
                          batch_encoding = image_processor.pad(
                              pixel_values,       # Pass the list directly
                              return_tensors="pt"
                          )
                          # --- End Change ---
                      except Exception as e_pad:
                          print(f"Error during image_processor.pad: {e_pad}")
                          shapes = [pv.shape for pv in pixel_values]
                          print(f"  Shapes of tensors passed to pad: {shapes}")
                          return None # Skip batch if padding fails
                      # Combine padded values/mask with the original list of label dicts
                      batch_encoding['labels'] = labels
                      return batch_encoding
                  # --- End Custom Collator Definition ---

                  print("Using custom object detection collator.")
                  print("Creating dataloaders (num_workers=0)...")
                  train_dataloader = DataLoader(train_dataset, collate_fn=custom_object_detection_collator, batch_size=FINE_TUNE_BATCH_SIZE, shuffle=True, num_workers=0)
                  val_dataloader = DataLoader(val_dataset, collate_fn=custom_object_detection_collator, batch_size=FINE_TUNE_BATCH_SIZE * 2, shuffle=False, num_workers=0)
                  print("Dataloaders created.")

             except Exception as e_load:
                  print(f"ERROR creating Datasets/Dataloaders: {e_load}")
                  traceback.print_exc(); DO_FINE_TUNING = False

# --- Load COCO GT data ---
if os.path.exists(coco_annotation_file):
     try: print(f"\nLoading COCO ground truth for mAP evaluation from: {coco_annotation_file}"); coco_gt = COCO(coco_annotation_file); print("COCO GT loaded.")
     except Exception as e_coco: print(f"ERROR loading COCO annotations file '{coco_annotation_file}': {e_coco}"); coco_gt = None
else: print(f"\nCOCO annotation file for evaluation not found at: {coco_annotation_file}"); coco_gt = None
# --- End COCO load ---

if not DO_FINE_TUNING: print("\nFine-tuning disabled due to errors or configuration.")


--- Preparing KITTI Dataset ---

Creating training and validation datasets...
Found 1954 total images.
Using 1758 train and 196 validation images.
Using label2id map for dataset: {'Car': 0, 'Pedestrian': 1, 'Cyclist': 2}
Dataset Initialized. Found 1758 images. Category map: {'car': 0, 'pedestrian': 1, 'cyclist': 2}
Dataset Initialized. Found 196 images. Category map: {'car': 0, 'pedestrian': 1, 'cyclist': 2}
Using custom object detection collator.
Creating dataloaders (num_workers=0)...
Dataloaders created.

COCO annotation file for evaluation not found at: /content/drive/MyDrive/kitti_subset/annotations.json


In [ ]:
# =============================================================================
# Cell 8: Fine-tuning Loop
# =============================================================================
print("\n--- Fine-tuning Pruned Model ---")

final_model_saved_path = None # Initialize path for final model

if not DO_FINE_TUNING:
    print("Skipping fine-tuning (disabled).")
elif 'model_pruned' not in locals() or model_pruned is None:
     print("Skipping fine-tuning: Pruned model is not available.")
elif train_dataloader is None or val_dataloader is None:
     print("Skipping fine-tuning: Dataloaders not available.")
else:
    print(f"Starting fine-tuning for {FINE_TUNE_EPOCHS} epochs...")
    model_pruned.to(device) # Ensure model is on GPU

    # Filter parameters that require gradients (only necessary if some layers were frozen)
    params_to_optimize = filter(lambda p: p.requires_grad, model_pruned.parameters())
    optimizer = torch.optim.AdamW(params_to_optimize, lr=FINE_TUNE_LR, weight_decay=1e-4)
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1) # Example scheduler

    print(f"Optimizer: AdamW, LR={FINE_TUNE_LR}, Scheduler: StepLR")
    if not list(filter(lambda p: p.requires_grad, model_pruned.parameters())):
         print("WARNING: No trainable parameters found in the model!")
         # Set DO_FINE_TUNING to False maybe? Or just let it run doing nothing.

    # Training loop
    for epoch in range(FINE_TUNE_EPOCHS):
        print(f"\n--- Epoch {epoch+1}/{FINE_TUNE_EPOCHS} ---")
        model_pruned.train() # Set model to training mode
        model_pruned.to(device) # Ensure model is on the correct device at epoch start
        total_train_loss = 0
        processed_batches = 0

        progress_bar_train = tqdm(train_dataloader, desc=f"Epoch {epoch+1} Training", leave=False)
        for batch in progress_bar_train:
            if batch is None: continue # Skip bad batches

            try:
                pixel_values = batch["pixel_values"].to(device)
                # pixel_mask = batch["pixel_mask"].to(device) # Usually handled by processor or model internals
                labels = [{k: v.to(device) for k, v in t.items()} for t in batch["labels"]]

                # Forward pass
                outputs = model_pruned(pixel_values=pixel_values, pixel_mask=None, labels=labels)
                loss = outputs.loss
                loss_dict = outputs.loss_dict

                if not torch.isfinite(loss):
                     print(f"WARNING: NaN/Inf loss detected ({loss.item()}). Skipping batch.")
                     optimizer.zero_grad() # Clear potential bad grads
                     continue

                # Backward pass and optimization
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model_pruned.parameters(), max_norm=0.1)
                optimizer.step()

                total_train_loss += loss.item()
                processed_batches += 1
                progress_bar_train.set_postfix({'loss': f"{loss.item():.4f}", 'avg_loss': f"{total_train_loss/processed_batches:.4f}"})

            except Exception as e_train:
                print(f"\nERROR during training batch: {e_train}")
                traceback.print_exc()
                continue # Continue to next batch

        avg_train_loss = total_train_loss / processed_batches if processed_batches > 0 else 0
        print(f"Epoch {epoch+1} Average Training Loss: {avg_train_loss:.4f}")

        # --- Validation ---
        model_pruned.eval() # Set model to evaluation mode
        total_val_loss = 0
        val_batches = 0
        print("Running validation...")
        progress_bar_val = tqdm(val_dataloader, desc=f"Epoch {epoch+1} Validation", leave=False)
        with torch.no_grad():
            for batch in progress_bar_val:
                 if batch is None: continue
                 try:
                      pixel_values = batch["pixel_values"].to(device)
                      labels = [{k: v.to(device) for k, v in t.items()} for t in batch["labels"]]

                      outputs = model_pruned(pixel_values=pixel_values, pixel_mask=None, labels=labels)
                      loss = outputs.loss

                      if torch.isfinite(loss):
                           total_val_loss += loss.item()
                           val_batches += 1
                 except Exception as e_val:
                      print(f"\nERROR during validation batch: {e_val}")
                      continue

        avg_val_loss = total_val_loss / val_batches if val_batches > 0 else 0
        print(f"Epoch {epoch+1} Average Validation Loss: {avg_val_loss:.4f}")

        # Step the scheduler
        lr_scheduler.step()
        print(f"Epoch {epoch+1} completed. Current LR: {optimizer.param_groups[0]['lr']:.2e}")

        # --- Save Checkpoint ---
        checkpoint_saved_this_epoch = False
        try:
            # Prefer saving full model state using save_pretrained in the checkpoint dir
            ckpt_dir = os.path.join(output_dir, f"checkpoint-epoch-{epoch+1}")
            print(f"  Saving checkpoint to: {ckpt_dir}")
            model_pruned.save_pretrained(ckpt_dir)
            # Also save processor config for easy reloading
            if image_processor: image_processor.save_pretrained(ckpt_dir)
            print(f"  Checkpoint saved successfully.")
            checkpoint_saved_this_epoch = True
        except Exception as e_ckpt:
             print(f"  ERROR saving checkpoint: {e_ckpt}")
             traceback.print_exc()

    print("\n--- Fine-tuning finished ---")

    # --- Save Final Model ---
    try:
        final_save_dir = os.path.join(output_dir, "final_model")
        print(f"Saving final model to: {final_save_dir}")
        model_pruned.save_pretrained(final_save_dir)
        if image_processor: image_processor.save_pretrained(final_save_dir)
        # Update the path variable for the summary cell
        final_model_saved_path = final_save_dir
        print("Final model saved successfully.")
    except Exception as e_final_save:
        print(f"Error saving final model: {e_final_save}")
        traceback.print_exc()

# Cleanup dataloaders to free memory
del train_dataloader, val_dataloader, train_dataset, val_dataset
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("Cleaned up dataloaders.")


--- Fine-tuning Pruned Model ---
Starting fine-tuning for 5 epochs...
Optimizer: AdamW, LR=1e-05, Scheduler: StepLR

--- Epoch 1/5 ---


Epoch 1 Training:   0%|          | 0/879 [00:00<?, ?it/s]

In [ ]:
# =============================================================================
# Cell 9: Evaluation (mAP after Fine-tuning)
# =============================================================================
print("\n--- Evaluating mAP After Fine-tuning (Requires COCO annotations) ---")

# --- Decide which model to evaluate ---
eval_model = None
eval_model_path = None

# Option 1: Evaluate the model currently in memory (if fine-tuning just finished)
if 'model_pruned' in locals() and model_pruned is not None and DO_FINE_TUNING:
     print("Evaluating the model currently in memory (fine-tuned)...")
     eval_model = model_pruned
# Option 2: Load the final saved model
elif 'final_model_saved_path' in locals() and final_model_saved_path and os.path.isdir(final_model_saved_path):
     print(f"Loading final saved model from: {final_model_saved_path}")
     try:
          # Need config and processor if loading from scratch
          config = AutoConfig.from_pretrained(final_model_saved_path)
          image_processor = AutoImageProcessor.from_pretrained(final_model_saved_path)
          eval_model = AutoModelForObjectDetection.from_pretrained(final_model_saved_path, config=config)
          print("Loaded final model successfully.")
     except Exception as e_load_final:
          print(f"ERROR loading final model: {e_load_final}")
          eval_model = None
# Option 3: Load the pruned structure (before fine-tuning)
elif 'pruned_model_saved_path' in locals() and pruned_model_saved_path and os.path.isdir(pruned_model_saved_path): # Check if it's a directory from save_pretrained
     print(f"Loading pruned structure (before fine-tuning) from: {pruned_model_saved_path}")
     try:
          config = AutoConfig.from_pretrained(pruned_model_saved_path)
          image_processor = AutoImageProcessor.from_pretrained(pruned_model_saved_path)
          eval_model = AutoModelForObjectDetection.from_pretrained(pruned_model_saved_path, config=config)
          print("Loaded pruned (pre-FT) model successfully.")
     except Exception as e_load_pruned:
          print(f"ERROR loading pruned model: {e_load_pruned}")
          eval_model = None
else:
     print("No pruned model available in memory or found in standard save locations.")

# --- Proceed with Evaluation if Model and GT are available ---
map_after = -1.0 # Initialize
map_50_after = -1.0

if eval_model is None:
    print("Model not available for evaluation.")
elif 'kitti_dataset_full' not in locals() or kitti_dataset_full is None or coco_gt is None:
    print("COCO dataset/annotations not loaded. Cannot calculate COCO mAP.")
else:
    print("Preparing for mAP evaluation...")
    eval_model.to(device) # Ensure model is on GPU
    eval_model.eval()

    # Need a DataLoader for the *full* dataset used for COCO eval
    # Use batch size 1 for inference unless you implement robust batch padding/handling
    eval_batch_size = 1 # Recommended for simplicity with varying image sizes
    # Define a simple collate for PIL images if bs=1
    def pil_collate(batch):
         # CocoDetection returns (PIL Image, target_dict)
         return batch[0] # Return (image, target) tuple for bs=1

    eval_loader = DataLoader(kitti_dataset_full, batch_size=eval_batch_size, shuffle=False, num_workers=0, collate_fn=pil_collate)

    coco_results_after = []
    processed_eval_count = 0

    with torch.no_grad():
        for batch_data in tqdm(eval_loader, desc="Evaluating mAP"):
            image, target = batch_data # Unpack tuple for bs=1
            image_id = target['image_id'].item() # Get original image ID

            try:
                # Preprocess image
                inputs = image_processor(images=image, return_tensors="pt").to(device)
                outputs = eval_model(**inputs)

                # Post-process
                target_sizes = torch.tensor([image.size[::-1]], device=device) # (height, width)
                results = image_processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.1) # Use a suitable threshold

                # Format results for COCO eval
                res = results[0]
                boxes = res["boxes"].cpu().numpy()
                scores = res["scores"].cpu().numpy()
                labels = res["labels"].cpu().numpy()

                for box, score, label in zip(boxes, scores, labels):
                    x_min, y_min, x_max, y_max = box.tolist()
                    coco_results_after.append({
                        "image_id": image_id,
                         # Ensure this category_id matches the IDs in your coco_gt annotations
                        "category_id": label.item(), # Use the raw label ID from the model head
                        "bbox": [x_min, y_min, x_max - x_min, y_max - y_min], # COCO format XYWH
                        "score": float(score)
                    })
                processed_eval_count += 1

            except Exception as e_infer:
                 print(f"\nError during inference/postprocessing for image_id {image_id}: {e_infer}")
                 # Optionally add traceback.print_exc()
                 continue # Skip this image

    print(f"\nProcessed {processed_eval_count} images for evaluation.")

    if coco_results_after:
        print("Running COCO evaluation API...")
        try:
            coco_dt = coco_gt.loadRes(coco_results_after)
            coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
            coco_eval.evaluate()
            coco_eval.accumulate()
            coco_eval.summarize()
            map_after = coco_eval.stats[0] # mAP @ IoU=0.50:0.95
            map_50_after = coco_eval.stats[1] # mAP @ IoU=0.50
            print(f"\n--- mAP Results ---")
            print(f"mAP @ IoU=0.50:0.95 (AP): {map_after:.4f}")
            print(f"mAP @ IoU=0.50 (AP50):     {map_50_after:.4f}")
        except Exception as e_eval:
            print(f"ERROR during COCO evaluation: {e_eval}")
            traceback.print_exc()
    else:
        print("No evaluation results generated to run COCO eval.")

# Cleanup
del eval_model, eval_loader, kitti_dataset_full, coco_gt
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("Cleaned up evaluation objects.")

In [ ]:
# =============================================================================
# Cell 10: Final Summary
# =============================================================================
print("\n--- Final Summary ---")
print(f"Pruning Ratio Target (per layer): {GLOBAL_PRUNING_RATIO:.2f}")

# --- Use calculated metrics if available ---
print(f"\nParameters:")
if 'original_params' in locals() and original_params > 0 : print(f"  Original (Post Head Resize): {original_params:,}")
else: print("  Original: (Not calculated/available)")
if 'pruned_params' in locals() and pruned_params >= 0 : print(f"  Pruned (Torch-Pruning):      {pruned_params:,}")
else: print("  Pruned:   (Not calculated/available)")

if ('original_params' in locals() and original_params > 0 and
    'pruned_params' in locals() and pruned_params >= 0):
     reduction = (original_params - pruned_params) / original_params * 100
     print(f"  Reduction:                   {reduction:.2f}%")

print(f"\nGFLOPs:")
if 'original_gflops' in locals() and original_gflops > 0: print(f"  Original: {original_gflops:.2f}")
else: print("  Original: (Not calculated/available)")
if 'pruned_gflops' in locals() and pruned_gflops >= 0 : print(f"  Pruned:   {pruned_gflops:.2f}")
else: print("  Pruned:   (Not calculated/available)")

if ('original_gflops' in locals() and original_gflops > 0 and
    'pruned_gflops' in locals() and pruned_gflops >= 0):
     gflops_reduction = (original_gflops - pruned_gflops) / original_gflops * 100
     print(f"  Reduction: {gflops_reduction:.2f}%")

# --- Report Saved File Locations ---
print(f"\nSaved Files in: {output_dir}")
pruned_structure_dir = os.path.join(output_dir) # save_pretrained saves to the dir
final_model_dir = os.path.join(output_dir, "final_model") # Default save_pretrained dir

if os.path.isdir(pruned_structure_dir) and os.path.exists(os.path.join(pruned_structure_dir, "model.safetensors")): # Check if save_pretrained likely worked
    print(f"  Pruned Structure saved via save_pretrained in: {pruned_structure_dir}")
elif 'pruned_model_saved_path' in locals() and pruned_model_saved_path and os.path.exists(pruned_model_saved_path): # Fallback check for direct state dict save
    print(f"  Pruned Structure state_dict: {os.path.basename(pruned_model_saved_path)} ({os.path.getsize(pruned_model_saved_path)/(1024*1024):.2f} MB)")
else:
    print("  Pruned Structure: Not saved or path not found.")

if DO_FINE_TUNING and os.path.isdir(final_model_dir) and os.path.exists(os.path.join(final_model_dir, "model.safetensors")):
     print(f"  Final Fine-tuned saved via save_pretrained in: {final_model_dir}")
elif DO_FINE_TUNING:
     print(f"  Final Fine-tuned: Not saved or path not found.")
else:
     print(f"  Final Fine-tuned: Fine-tuning skipped.")

# --- Report mAP ---
print(f"\nmAP Evaluation Results:")
if 'map_after' in locals() and map_after >= 0: # Check if eval ran and produced valid result
    print(f"  mAP @ IoU=0.50:0.95 (AP): {map_after:.4f}")
    print(f"  mAP @ IoU=0.50 (AP50):     {map_50_after:.4f}")
else:
    print("  mAP evaluation not performed or failed.")
    if not os.path.exists(coco_annotation_file): print("  (COCO annotation file was missing)")
    elif coco_gt is None: print("  (COCO GT object failed to load)")


print("\n--- End of Notebook ---")